# PdrQC Tiny-MNIST Execution Demo

Objective: verify that the compact paper code runs end to end on the included small MNIST subset.

This notebook is an execution check, not a reproduction of the full-data paper values. It runs only PdrQC and the comparison methods explicitly described in the paper.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

SEED = 42
candidate_roots = [Path.cwd(), Path.cwd() / "paper" / "PdrQC" / "code"]
CODE_ROOT = next(
    path.resolve()
    for path in candidate_roots
    if (path / "pdrqc_paper_experiments.py").exists()
)
RUN_DIR = CODE_ROOT / "results" / "notebook_runs" / "01_pdrqc_tiny_mnist_demo"
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON = RUN_DIR / "tiny_paper_demo.json"
print(f"CODE_ROOT={CODE_ROOT}")
print(f"OUTPUT_JSON={OUTPUT_JSON}")

## Plan

- Use only digits 0, 1, and 2 from the included processed subset.
- Use four qubits and Pauli weight at most two.
- Run two progressive upload-search rounds.
- Evaluate PdrQC, PCA-LR, Full Pauli, computational-basis readout, DRC, and QKC.
- Keep DRC epochs and finite-shot repeats small so this remains a quick check.

In [ ]:
command = [
    sys.executable,
    "pdrqc_paper_experiments.py",
    "--dataset", "mnist",
    "--classes", "0,1,2",
    "--n-qubits", "4",
    "--max-pauli-weight", "2",
    "--max-rounds", "2",
    "--train-per-class", "20",
    "--validation-per-class", "5",
    "--test-per-class", "10",
    "--shot-budget", "2048",
    "--shot-repeats", "3",
    "--drc-epochs", "5",
    "--seed", str(SEED),
    "--output", str(OUTPUT_JSON),
]
completed = subprocess.run(
    command,
    cwd=CODE_ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print("Experiment completed.")
print(completed.stdout.splitlines()[-1])

## Compact Results

The next cell reports only the principal accuracy and measurement-cost fields. Small-sample values are expected to differ from the paper.

In [ ]:
payload = json.loads(OUTPUT_JSON.read_text())
rows = [
    {
        "method": "pdrqc",
        "ideal_accuracy": payload["pdrqc"]["ideal_test_accuracy"],
        "finite_shot_accuracy": payload["pdrqc"]["finite_shot_test_mean"],
        "shots_per_sample": payload["pdrqc"]["actual_shots_per_sample"],
    }
]
for item in payload["paper_readout_baselines"]:
    rows.append({
        "method": item["method"],
        "ideal_accuracy": item["ideal_test_accuracy"],
        "finite_shot_accuracy": None,
        "shots_per_sample": None,
    })
for item in payload["quantum_classifier_baselines"]:
    rows.append({
        "method": item["method"],
        "ideal_accuracy": item["ideal_test_accuracy"],
        "finite_shot_accuracy": item["finite_shot_test_mean"],
        "shots_per_sample": item["actual_shots_per_sample"],
    })
summary = pd.DataFrame(rows)
summary

## QKC Measurement-Cost Check

PdrQC divides one total shot budget among its commuting Pauli groups. QKC instead assigns the shot setting to every support-vector kernel value, so its total cost scales with the support-vector count.

In [ ]:
qkc = next(
    item
    for item in payload["quantum_classifier_baselines"]
    if item["method"] == "quantum_kernel"
)
cost_check = {
    "support_vectors": qkc["support_vector_count"],
    "shots_per_kernel_value": qkc["shots_per_kernel_value"],
    "qkc_shots_per_sample": qkc["actual_shots_per_sample"],
    "pdrqc_shots_per_sample": payload["pdrqc"]["actual_shots_per_sample"],
    "qkc_to_pdrqc_ratio": (
        qkc["actual_shots_per_sample"]
        / payload["pdrqc"]["actual_shots_per_sample"]
    ),
}
cost_check

## Interpretation Boundary

- The included subset is intentionally small.
- Finite-shot sampling is not a hardware-noise simulation.
- Use the full public MNIST/Fashion-MNIST data and the paper settings for a paper-scale rerun.
- The notebook executes only methods and evaluation mechanisms described in the paper.